# RSNA Knee — Training

Thin shell: pulls `knee` from GitHub at a pinned commit, trains on the pre-mounted
competition data, exports checkpoints to `/kaggle/working`.

Current experiments: **E003-blended-labels + E004-dinov2-frozen, one run.** Both
arms train on all 4,407 studies with the blended soft labels (`knee-labels`
dataset): E003 = frozen ResNet-34 @224 (labels lever, LB A/B vs E002), E004 =
frozen DINOv2 ViT-S/14 @518 (backbone lever, CV A/B vs the E003 arm — same labels,
seed, and folds, so the CV delta is attributable to the backbone). Each arm fills
its own feature bank (persisted for cheap retrains), fits heads, and runs the
pooled-OOF CV. Ship the winning arm's checkpoints. Requires the `WANDB_API_KEY`
Kaggle secret and internet (DINOv2 weights download at train time).

In [ ]:
# Pin a commit so every checkpoint traces to exact code. Training notebooks have internet.
# --no-deps everywhere: Kaggle's image already ships torch/timm/sklearn/numpy compiled
# together; letting pip resolve our pins upgrades numpy and breaks the whole stack.
# STALE for E004 — bump to the dinov2-backbone squash merge before `kaggle kernels push`
# (this run needs DINOV2_BACKBONE and train_blended's backbone kwarg, absent at 970b1df).
COMMIT = "970b1df"
%pip install -q --no-deps "git+https://github.com/Josie29/capstone-rsna-knee@{COMMIT}#egg=knee"
%pip install -q --no-deps pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg pylibjpeg-rle

import numpy  # fail fast if the image stack is broken or missing
import timm
import torch

print("numpy", numpy.__version__, "| torch", torch.__version__, "| timm", timm.__version__)

from knee.series import SeriesType
from knee.train_blended import BLENDED_LABEL_SOURCE, train_blended

In [ ]:
# Competition data (DICOMs + series metadata) is pre-mounted; the blended soft
# labels arrive via the attached private knee-labels dataset (issue #2 resolved).
from pathlib import Path

from knee.data import load_blended_labels

SLUG = "rsna-knee-abnormality-detection"
# Kaggle mounts competitions under /kaggle/input/competitions/<slug> (newer layout)
# or /kaggle/input/<slug> (older docs/examples); accept either.
candidates = [Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG]
COMP_ROOT = next((p for p in candidates if (p / "train.csv").exists()), None)
if COMP_ROOT is None:
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"competition data not found; mounts: {listing}")
print("competition root:", COMP_ROOT)

# Attached datasets moved mount points too; probe the plausible locations.
LABELS_CSV = "blended_labels_v1.csv"
label_candidates = [
    Path("/kaggle/input/knee-labels") / LABELS_CSV,
    Path("/kaggle/input/datasets/josiemachalek/knee-labels") / LABELS_CSV,
    Path("/kaggle/input/datasets/knee-labels") / LABELS_CSV,
]
labels_path = next((p for p in label_candidates if p.exists()), None)
if labels_path is None:
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"{LABELS_CSV} not found; mounts: {listing}")
labels = load_blended_labels(labels_path)
print(f"blended labels: {len(labels)} studies from {labels_path}")

In [ ]:
# Metrics/hyperparams only -- no report text, no StudyInstanceUIDs (rule 2.4.b).
# Best-effort: the run proceeds without wandb if the secret isn't configured.
run = None
try:
    import wandb
    from kaggle_secrets import UserSecretsClient

    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    run = wandb.init(
        project="rsna-knee",
        config={"commit": COMMIT, "label_source": BLENDED_LABEL_SOURCE, "n_label_studies": len(labels)},
    )
except Exception as exc:  # noqa: BLE001 — telemetry must never kill a training run
    print(f"wandb disabled: {exc}")

In [ ]:
# Two arms, identical except backbone + its native input size (518 is the ViT's
# fixed patch-14 x 37 input — never change one without the other). Same ensemble
# shape and strict series typing as E001/E002. Forum context in
# docs/rsna_brain.md §3.5: the E004 arm decides whether backbone investment
# continues (unfreezing next) or stops in favor of input-geometry work.
from knee.model import DEFAULT_BACKBONE, DINOV2_BACKBONE

SERIES_TYPES = [SeriesType.SAGITTAL_FLUID, SeriesType.CORONAL_FLUID, SeriesType.AXIAL_FLUID]
# tag -> (timm backbone, input size); tag names the checkpoint subdir and bank file.
ARMS = {
    "resnet34": (DEFAULT_BACKBONE, 224),   # E003
    "dinov2": (DINOV2_BACKBONE, 518),      # E004
}

CHECKPOINT_DIR = Path("/kaggle/working")

def bank_path(tag: str) -> Path:
    return CHECKPOINT_DIR / f"feature_bank_{BLENDED_LABEL_SOURCE}_{tag}.pt"

In [ ]:
# Per arm: one threaded decode pass over all 4.4k studies fills the bank (the
# hours — the 518px ViT arm is the long pole; budget most of a T4 session for the
# two passes), then three head fits from cached features (the seconds). Checkpoints
# go to a per-arm subdir — both arms share {label_source}_{plane}.pt filenames, and
# the backbone recorded inside the checkpoint is what disambiguates them.
from knee.cv import save_feature_bank

banks, arm_results = {}, {}
for tag, (backbone, input_size) in ARMS.items():
    print(f"=== arm {tag}: {backbone} @ {input_size}px ===")
    bank, results = train_blended(
        COMP_ROOT,
        labels,
        CHECKPOINT_DIR / tag,
        series_types=SERIES_TYPES,
        backbone=backbone,
        input_size=input_size,
    )
    save_feature_bank(bank, bank_path(tag))
    banks[tag], arm_results[tag] = bank, results
    print("plane coverage:", {t.value: n for t, n in bank.plane_coverage().items()})
    for result in results:
        print(f"{result.series_type.value}: trained on {result.n_studies} studies")
        # In-sample vs thresholded blended labels: proves the features carry signal
        # against the miner's labels; the generalization number is the CV below.
        print(result.in_sample_auc)

In [ ]:
# Local eval: pooled out-of-fold stratified CV of the full ensemble over ALL blended
# studies, per arm. Same labels + seed means identical fold assignments across arms,
# so the macro delta is attributable to the backbone (decision rule in experiments.md
# E004: ship dinov2 only if its macro beats resnet34's by more than the per-repeat
# spread). Caveat: labels are miner-derived, so these AUCs measure agreement with the
# report miner — a model-selection signal, with the LB as truth check.
from knee.cv import cross_validate

cvs = {}
for tag, bank in banks.items():
    print(f"=== arm {tag} ===")
    cvs[tag] = cv = cross_validate(bank)
    print(f"macro OOF AUC {cv.macro_auc:.3f} over {cv.n_repeats} repeats: "
          + ", ".join(f"{m:.3f}" for m in cv.macro_auc_per_repeat))
    print({label: round(auc, 3) for label, auc in cv.per_label_auc.items()})

In [ ]:
# Checkpoints + feature banks are in /kaggle/working, which persists as notebook
# output; publish the WINNING arm's three .pt checkpoints (subdir resnet34/ or
# dinov2/) as a new knee-weights dataset VERSION that REPLACES the previous ones —
# inference rejects duplicate series types, so checkpoints from two arms or two
# versions must never be mounted together.
import math

if run is not None:
    run.config.update({"arms": {tag: {"backbone": b, "input_size": s} for tag, (b, s) in ARMS.items()}})
    for tag, results in arm_results.items():
        wandb.log(
            {
                f"in_sample_auc/{tag}/{result.series_type.value}/{label}": auc
                for result in results
                for label, auc in result.in_sample_auc.items()
                if not math.isnan(auc)
            }
        )
    for tag, cv in cvs.items():
        wandb.log(
            {
                f"cv/macro_auc/{tag}": cv.macro_auc,
                **{f"cv/auc/{tag}/{label}": auc for label, auc in cv.per_label_auc.items() if not math.isnan(auc)},
            }
        )
    run.finish()